### Baseline

In [ ]:
# Configuración de recursos según la máquina
entorno = 'casa'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

#### Librerías y Path Variables

In [ ]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # CAMBIO: ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # CAMBIO: carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_meses.parquet'
ruta_meses = dataset_path + dataset_file
print(ruta_meses, Path(ruta_meses).exists())

In [ ]:
%%time
# Conversión única: csv.gz original -> parquet (mismo contenido, todos los meses y columnas)
ruta_parquet = dataset_path + 'competencia_02_crudo.parquet'

if Path(ruta_parquet).exists():
    print('El parquet ya existe, no se regenera:', ruta_parquet)
else:
    con_conv = duckdb.connect()
    con_conv.execute(f"SET threads = {duckdb_threads}")
    con_conv.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
    con_conv.execute(f"""
        COPY (SELECT * FROM read_csv('{ruta_crudo}', sample_size = -1))
        TO '{ruta_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print('listo:', round(Path(ruta_parquet).stat().st_size / 1e6), 'MB')

# Columnas que quedaron como texto (las 0/1 como tmobile_app hay que pasarlas a número)
con_chk = duckdb.connect()
print('Columnas de texto:', [r[0] for r in con_chk.execute(f"SELECT column_name FROM (DESCRIBE SELECT * FROM read_parquet('{ruta_parquet}')) WHERE column_type = 'VARCHAR'").fetchall()])

#### Carga del 2019
Se lee el `.csv.gz` original **una sola vez** y se deja una tabla en memoria de DuckDB (`d2019`). No se guarda ningún subset en disco: si se reinicia el kernel, se vuelve a correr esta celda.

In [ ]:
%%time
# Conexión DuckDB con los recursos de la máquina
con = duckdb.connect()
con.execute(f"SET threads = {duckdb_threads}")
con.execute(f"SET memory_limit = '{duckdb_memory_limit}'")

MES_DESDE, MES_HASTA = 201901, 201912

# sample_size=-1: DuckDB mira TODO el archivo para decidir el tipo de cada columna (más lento, pero evita
# que una columna vacía al principio quede mal tipada). 
con.execute(f"""
    CREATE OR REPLACE TABLE d2019 AS
    SELECT * FROM read_parquet('{ruta_meses}')            
    WHERE foto_mes BETWEEN {MES_DESDE} AND {MES_HASTA}
""")
print(con.execute("SELECT count(*) FROM d2019").fetchall()[0][0], 'filas')

#### Filas, Clientes, Variables y Clase Ternaria por Mes

In [ ]:
mes_resumen = con.sql("""
    SELECT foto_mes,
           count(*)                                      AS filas,
           count(DISTINCT numero_de_cliente)             AS clientes,
           count(*) - count(DISTINCT numero_de_cliente)  AS duplicados      -- tiene que dar 0
    FROM d2019
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df()
mes_resumen

In [ ]:
import math

cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE d2019)").fetchall()]
print(len(cols), 'columnas')
print('clase_ternaria' in cols)

n_col = 5                                          # cantidad de columnas a mostrar
n_fil = math.ceil(len(cols) / n_col)
relleno = cols + [''] * (n_fil * n_col - len(cols))     # completa con vacíos para que cierre el rectángulo
tabla = pd.DataFrame(np.array(relleno).reshape(n_col, n_fil).T)
tabla.style.hide(axis='index').hide(axis='columns')

In [ ]:
# Clase por mes. Esperado: 201911 solo con BAJA+1, 201912 sin clase
clase_mes = (con.sql("""
    SELECT foto_mes, coalesce(clase_ternaria, '(sin clase)') AS clase, count(*) AS filas
    FROM d2019
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df().pivot(index='foto_mes', columns='clase', values='filas').fillna(0).astype(int))

total = clase_mes.sum(axis=1)
clase_mes['pct_BAJA+2'] = (clase_mes.get('BAJA+2', 0) / total * 100).round(3)
clase_mes

#### Tipos de Variables

In [ ]:
esquema = con.sql("DESCRIBE d2019").df()[['column_name', 'column_type']]
ids = ['numero_de_cliente', 'foto_mes']

tipos_num = ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT', 'HUGEINT', 'UTINYINT', 'USMALLINT', 'UINTEGER', 'UBIGINT',
             'FLOAT', 'DOUBLE', 'DECIMAL')
es_num = esquema['column_type'].str.startswith(tipos_num)

cols_num = [c for c in esquema.loc[es_num, 'column_name'] if c not in ids]            # a las que se les mide nulos y ceros
cols_otras = [c for c in esquema.loc[~es_num, 'column_name'] if c not in ids]         # texto, fechas, etc.

print(len(esquema), 'columnas |', len(cols_num), 'numéricas |', len(cols_otras), 'no numéricas')
print('No numéricas:', cols_otras)           # clase_ternaria y, si aparecen, columnas 0/1 que llegaron como texto
esquema['column_type'].value_counts()

#### Variables Rotas
Se calcula todo en una sola pasada sobre `d2019`. Los dos porcentajes son sobre **todas las filas del mes** (un cero y un nulo no se pisan).

##### % Nulos y Ceros Globales

In [ ]:
%%time
# Una sola consulta: por mes, % de nulos y % de ceros de cada variable numérica
exprs = []
for c in cols_num:
    exprs.append(f'avg(("{c}" IS NULL)::int)                  AS "{c}__nul"')
    exprs.append(f'avg(coalesce(("{c}" = 0), false)::int)     AS "{c}__cero"')       # NULL no cuenta como cero

res = con.sql(f"""
    SELECT foto_mes, {', '.join(exprs)}
    FROM d2019
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df().set_index('foto_mes')

# Tablas variable x mes, en porcentaje
pct_nulos = (res[[f'{c}__nul' for c in cols_num]] * 100).T
pct_ceros = (res[[f'{c}__cero' for c in cols_num]] * 100).T
pct_nulos.index = pct_ceros.index = pd.Index(cols_num, name='variable')

# Las no numéricas: solo nulos
if cols_otras:
    otras = con.sql(f"""
        SELECT foto_mes, {', '.join(f'avg(("{c}" IS NULL)::int) * 100 AS "{c}"' for c in cols_otras)}
        FROM d2019 GROUP BY foto_mes ORDER BY foto_mes
    """).df().set_index('foto_mes').T
    print('Nulos (%) en columnas no numéricas:')
    display(otras.round(2))

print(pct_nulos.shape)

# CAMBIO: mostrar las dos tablas completas (todas las variables, sin cortar)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
    print('\n% de NULOS por variable y mes:')
    display(pct_nulos.round(1))
    print('\n% de CEROS por variable y mes:')
    display(pct_ceros.round(1))

##### Variables sin Información en todo el 2019

In [ ]:
# Siempre vacías: en TODOS los meses, nulo o cero el 99,9% o más de las filas
vacio = pct_nulos + pct_ceros
siempre_vacias = vacio.index[vacio.min(axis=1) >= 99.9]
print(len(siempre_vacias), 'variables siempre vacías (nulo o cero):')
print(list(siempre_vacias))

# Solo nulos o solo ceros en todo el año
print('\nSiempre 100% nulas      :', list(pct_nulos.index[pct_nulos.min(axis=1) >= 99.9]))
print('Siempre 100% en cero    :', list(pct_ceros.index[pct_ceros.min(axis=1) >= 99.9]))

In [ ]:
vars_vacias = ['mcuenta_corriente_adicional', 'Master_madelantodolares', 'Visa_madelantodolares']

print('% NULOS por mes')
display(pct_nulos.loc[vars_vacias].round(1))

print('% CEROS por mes')
display(pct_ceros.loc[vars_vacias].round(1))

##### Diferencia en % de NULOS y CEROS vs 201903

In [ ]:
mes_ref = 201903

def diferencias(pct, mes_ref):
    dif = pct.sub(pct[mes_ref], axis=0).abs()   # |% mes - % mes_ref|
    dif = dif.drop(columns=mes_ref)
    dif['max_diff'] = dif.max(axis=1)
    return dif.sort_values('max_diff', ascending=False)

dif_nulos = diferencias(pct_nulos, mes_ref)
dif_ceros = diferencias(pct_ceros, mes_ref)

print('Diferencia en % de NULOS vs', mes_ref)
display(dif_nulos.head(20).round(1))
print('Diferencia en % de CEROS vs', mes_ref)
display(dif_ceros.head(20).round(1))

# Para ver una variable/mes puntual contra la referencia
def ver_mes(pct, mes, n=15):
    t = pd.DataFrame({f'ref {mes_ref}': pct[mes_ref], f'mes {mes}': pct[mes]})
    t['diff'] = (t.iloc[:, 1] - t.iloc[:, 0]).abs()
    return t.sort_values('diff', ascending=False).head(n).round(2)

##### Mapas de calor de las variables marcadas

Cada fila es una variable y cada columna un mes de 2019. El color muestra el **% de nulos** (primer mapa) o el **% de ceros** (segundo mapa) de esa variable en ese mes: blanco = 0%, rojo oscuro = 100%.

Las variables que se grafican son las 30 con mayor diferencia respecto del mes de referencia (201903), según el ranking `dif_nulos` / `dif_ceros` de la celda anterior. Por eso es el mismo ranking, pero visto en porcentajes reales: permite ubicar **en qué mes cambia** cada variable (por ejemplo, 0% hasta junio y 40% desde julio), cosa que en la tabla de diferencias no se ve directamente.

Cómo leerlo: una fila de color parejo en todos los meses es una variable estable. Un salto de color a mitad de año es un cambio de comportamiento que hay que investigar con `perfil(var)`.

In [ ]:
def mapa(pct, titulo, vars_):
    if len(vars_) == 0:
        print(titulo, '-> sin variables marcadas')
        return
    z = pct.loc[vars_]
    # CAMBIO: etiquetas tipo ene-19, horizontales
    meses = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
    etiquetas = [f"{meses[int(str(m)[4:6]) - 1]}-{str(m)[2:4]}" for m in z.columns]

    fig = px.imshow(z.values, x=etiquetas, y=list(z.index),
                    color_continuous_scale='Reds', zmin=0, zmax=100, aspect='auto',
                    text_auto='.0f', title=titulo)
    fig.update_xaxes(type='category', side='top', tickmode='array',
                     tickvals=etiquetas, tickangle=0)
    fig.update_yaxes(type='category', tickmode='array', tickvals=list(z.index))
    fig.update_layout(height=max(300, 22 * len(vars_) + 120), template='plotly_white')
    fig.show()

# CAMBIO: variables tomadas del ranking de diferencias vs mes de referencia (ya no de `rotas`)
vars_nulos = list(dif_nulos.head(30).index)
vars_ceros = list(dif_ceros.head(30).index)

mapa(pct_nulos, '% de nulos por mes (top 30 por diferencia vs 201903)', vars_nulos)
mapa(pct_ceros, '% de ceros por mes (top 30 por diferencia vs 201903)', vars_ceros)

**Mirar una Variable en Detalle**

In [ ]:
##### Mirar una variable en detalle
def perfil(var):
    """Por mes: nulos, ceros y resumen de los valores distintos de cero."""
    return con.sql(f"""
        SELECT foto_mes,
               count(*)                                                 AS filas,
               round(100 * avg(("{var}" IS NULL)::int), 2)              AS pct_nulos,
               round(100 * avg(coalesce(("{var}" = 0), false)::int), 2) AS pct_ceros,
               min("{var}") FILTER (WHERE "{var}" <> 0)                 AS min_nz,
               round(avg("{var}") FILTER (WHERE "{var}" <> 0), 2)       AS media_nz,
               median("{var}") FILTER (WHERE "{var}" <> 0)              AS mediana_nz,
               max("{var}")                                             AS max
        FROM d2019
        GROUP BY foto_mes ORDER BY foto_mes
    """).df()

perfil('ccajas_depositos')       # cambiar por la variable que interese

##### Guardar los resultados (para armar después la lista de variables a excluir o tratar)

In [ ]:
os.makedirs(out_path, exist_ok=True)
pct_nulos.round(3).to_csv(out_path + 'eda2019_pct_nulos.csv')
pct_ceros.round(3).to_csv(out_path + 'eda2019_pct_ceros.csv')
dif_nulos.round(3).to_csv(out_path + 'eda2019_dif_nulos.csv')   # CAMBIO
dif_ceros.round(3).to_csv(out_path + 'eda2019_dif_ceros.csv')   # CAMBIO
print('guardado en', out_path)